# UD2.07. Robustez, caché, coste y despliegue

**Módulo 5073 · Programación de Inteligencia Artificial · UD2 · Proyecto PR2**

Lo que separa un ejercicio de algo que se puede poner delante de una persona.

Cuatro cosas, y las cuatro se corrigen en PR2:

1. **Reintentos** con espera creciente, solo para los fallos temporales.
2. **Caché**, que aquí no es una optimización: es lo que impide que la factura se dispare.
3. **Coste**, calculado antes de construir y medido después.
4. **Despliegue**, con las claves fuera de la imagen.

In [ ]:
!pip install -q requests

In [ ]:
import time

import requests

## 1. Reintentos: solo 429 y 5xx

Un 400 reintentado sigue siendo un 400, y cada intento consume cuota. Los únicos códigos que
mejoran esperando son el **429** (vas demasiado rápido) y los **5xx** (el proveedor tiene un
problema pasajero).

La espera crece de forma exponencial. Y si el servicio manda la cabecera `Retry-After`, se
respeta: **el proveedor sabe mejor que tú cuándo volver**.

In [ ]:
REINTENTABLES = {429, 500, 502, 503, 504}


class ErrorServicio(Exception):
    """Fallo al hablar con un servicio remoto."""


def peticion_con_reintentos(metodo, url, *, intentos=4, base=1.5, timeout=15,
                            registro=None, **kwargs):
    """Hace la petición reintentando solo los fallos temporales.

    registro: lista opcional donde se anotan los tiempos de espera, para poder
    verlos en el cuaderno sin imprimir desde dentro de la función.
    """
    for intento in range(intentos):
        try:
            respuesta = requests.request(metodo, url, timeout=timeout, **kwargs)
        except requests.RequestException as error:
            if intento == intentos - 1:
                raise ErrorServicio(f"No se pudo contactar: {error}") from error
            espera = base ** intento
            if registro is not None:
                registro.append((intento + 1, "red", round(espera, 2)))
            time.sleep(espera)
            continue

        if respuesta.status_code not in REINTENTABLES:
            return respuesta

        if intento == intentos - 1:
            return respuesta                     # que decida quien llama

        # Retry-After manda sobre nuestro cálculo
        espera = float(respuesta.headers.get("Retry-After", base ** intento))
        if registro is not None:
            registro.append((intento + 1, respuesta.status_code, round(espera, 2)))
        time.sleep(espera)

    return respuesta

Fíjate en el `registro`: la función **no imprime nada**. Anota en una lista que le pasan desde
fuera, y quien llama decide qué hacer con ella. Un módulo de servicio que imprime por pantalla no
se puede usar dentro de una aplicación web ni dentro de una prueba automática.

Vamos a verlo funcionar **sin gastar cuota de ningún servicio de pago**: `httpbin.org` devuelve el
código que se le pida.

In [ ]:
registro = []
inicio = time.monotonic()

respuesta = peticion_con_reintentos(
    "GET", "https://httpbin.org/status/503", intentos=4, base=1.5, registro=registro
)

print(f"Código final: {respuesta.status_code}")
print(f"Tiempo total: {time.monotonic() - inicio:.1f} s")
print("\nintento  causa  espera (s)")
for fila in registro:
    print(f"   {fila[0]}     {fila[1]}     {fila[2]}")

Tres esperas de 1, 1,5 y 2,25 segundos antes de rendirse. Ahora el caso que **no** debe
reintentarse:

In [ ]:
registro = []
inicio = time.monotonic()

respuesta = peticion_con_reintentos(
    "GET", "https://httpbin.org/status/400", intentos=4, registro=registro
)

print(f"Código final: {respuesta.status_code}")
print(f"Tiempo total: {time.monotonic() - inicio:.1f} s")
print(f"Reintentos:   {len(registro)}")

Cero reintentos y respuesta inmediata. Si tu código reintenta un 400 cuatro veces, has convertido
un fallo instantáneo en cinco segundos de espera y cuatro llamadas facturadas, sin arreglar nada.

### La sesión reutilizada

Abrir una conexión HTTPS cuesta tiempo: negociación TLS incluida. Una `Session` de `requests`
reutiliza la conexión entre peticiones al mismo host.

In [ ]:
def mide(fn, veces=5):
    inicio = time.monotonic()
    for _ in range(veces):
        fn()
    return time.monotonic() - inicio


sesion = requests.Session()

sin_sesion = mide(lambda: requests.get("https://httpbin.org/get", timeout=15))
con_sesion = mide(lambda: sesion.get("https://httpbin.org/get", timeout=15))

print(f"Cinco peticiones sin sesión: {sin_sesion:.2f} s")
print(f"Cinco peticiones con sesión: {con_sesion:.2f} s")

La diferencia crece con el número de llamadas y con la latencia hasta el servicio. En una
aplicación que hace tres llamadas por interacción, se nota.

En Streamlit, la sesión se guarda con `@st.cache_resource`, que es exactamente para esto:
objetos vivos que se comparten, no datos que se copian.

## 2. Caché: aquí es dinero

Recuerda el modelo de ejecución de Streamlit: **cada interacción reejecuta el script entero**. Sin
caché, mover un deslizador que no tiene nada que ver con el análisis vuelve a llamar al API y
vuelve a pagar.

```python
@st.cache_data(ttl=3600, show_spinner=False)
def analiza_sentimiento(texto: str, idioma: str) -> dict:
    """El mismo texto no se envía dos veces al servicio en una hora."""
    ...

@st.cache_resource
def sesion_http() -> requests.Session:
    """Una sola sesión reutilizada por toda la aplicación."""
    return requests.Session()
```

| | `st.cache_data` | `st.cache_resource` |
|---|---|---|
| Para qué | Resultados: dict, listas, DataFrames | Objetos vivos: conexiones, clientes, modelos |
| Qué hace | Guarda una **copia** por cada juego de argumentos | Comparte **el mismo objeto** entre todas las sesiones |
| Si lo confundes | Objetos compartidos que se pisan entre usuarios | Copias caras de algo que no debía copiarse |

Y una condición que se olvida: los argumentos de una función con `@st.cache_data` tienen que ser
**hashables**. Pasarle una lista funciona en las versiones recientes, pero pasarle un objeto
grande o no serializable falla. Pasa cadenas y tuplas.

### Medir lo que ahorra

En la memoria de PR2 hay que decir **cuánto ahorra la caché**, medido. Aquí está el patrón, con
una caché mínima escrita a mano para que se vea el mecanismo sin depender de Streamlit.

In [ ]:
import functools

LLAMADAS = {"reales": 0, "servidas_de_cache": 0}


def cachea(fn):
    """Versión mínima de lo que hace @st.cache_data."""
    almacen = {}

    @functools.wraps(fn)
    def envoltorio(*args):
        if args in almacen:
            LLAMADAS["servidas_de_cache"] += 1
            return almacen[args]
        LLAMADAS["reales"] += 1
        almacen[args] = fn(*args)
        return almacen[args]

    return envoltorio


@cachea
def analiza(texto):
    """Simula una llamada al servicio: lenta y facturable."""
    time.sleep(0.3)
    return {"sentimiento": "positivo", "longitud": len(texto)}


# Un usuario que analiza tres textos y luego reordena una tabla cinco veces:
# el script se reejecuta ocho veces, pero los textos son siempre los mismos.
TEXTOS = ["primer texto", "segundo texto", "tercer texto"]

inicio = time.monotonic()
for _ in range(8):
    for texto in TEXTOS:
        analiza(texto)
duracion = time.monotonic() - inicio

print(f"Llamadas reales al servicio: {LLAMADAS['reales']}")
print(f"Servidas desde la caché:     {LLAMADAS['servidas_de_cache']}")
print(f"Tiempo total:                {duracion:.1f} s")
print(f"Sin caché habrían sido:      {24 * 0.3:.1f} s y 24 llamadas facturadas")

Tres llamadas en lugar de veinticuatro. Ese es el número que hay que poner en la memoria, y por
eso la caché no es una optimización opcional en una aplicación que paga por llamada.

Un aviso sobre el `ttl`: una caché sin caducidad guarda para siempre resultados de un servicio que
puede cambiar de versión de modelo. Una hora es un valor razonable para esta unidad.

Y **no caches datos personales sin pensarlo**. La caché es una copia más de esos datos, con su
propio tiempo de vida.

## 3. Coste: estimar antes, medir después

```
coste mensual  =  volumen × unidades por operación × precio por unidad
```

Lo difícil no es la fórmula: es que **cada servicio factura en una unidad distinta**.

| Servicio | Unidad |
|---|---|
| Language | Registros de texto de 1.000 caracteres |
| Vision | Transacciones (una por imagen y grupo de características) |
| Speech (voz a texto) | Horas de audio |
| Speech (texto a voz) | Caracteres sintetizados |
| Translator | Caracteres traducidos |

In [ ]:
def coste_mensual(operaciones_dia, caracteres_operacion, precio_por_mil_registros):
    """Coste de un servicio que factura por registros de 1.000 caracteres."""
    registros_operacion = max(1, -(-caracteres_operacion // 1000))   # redondeo hacia arriba
    registros_mes = operaciones_dia * 30 * registros_operacion
    return registros_mes / 1000 * precio_por_mil_registros, registros_mes


PRECIO = 1.0     # EUR por cada 1.000 registros. Comprueba la tarifa vigente.

for operaciones in (100, 500, 1500):
    coste, registros = coste_mensual(operaciones, caracteres_operacion=500,
                                     precio_por_mil_registros=PRECIO)
    print(f"{operaciones:5} operaciones/día -> {registros:8} registros/mes -> {coste:7.2f} EUR/mes")

Fíjate en el redondeo hacia arriba: **un texto de 500 caracteres factura un registro entero**,
igual que uno de 999. Agrupar documentos cortos en una sola petición no ahorra registros, pero sí
ahorra llamadas, que es lo que topa con el límite de peticiones por segundo.

Y haz siempre el escenario de que el volumen **se triplique**. Es donde muchas decisiones se dan
la vuelta, y es lo que se pide en A2.1 y en la memoria de PR2.

Dos costumbres que no cuestan nada:

- **Presupuesto con alerta** en el portal del proveedor, desde el primer día.
- **Contador de llamadas** en tu propio código. Un bucle mal escrito contra un servicio de pago es
  un accidente caro y silencioso.

## 4. Despliegue

| Opción | Ventaja | Inconveniente |
|---|---|---|
| **Streamlit Community Cloud** | Gratis, desde GitHub, en minutos | Recursos limitados, repositorio público |
| **Docker** | Idéntico en todas partes, portable | Hay que aprender contenedores |
| **Azure App Service o Container Apps** | Producción real, escalado, red privada | Cuesta dinero y configuración |

### Streamlit Community Cloud

1. Repositorio en GitHub con `requirements.txt` y el fichero de entrada.
2. En `share.streamlit.io`, *Create app*, y se elige `app/main.py`.
3. Los secretos, en *Advanced settings → Secrets*, con el mismo contenido que tendría
   `.streamlit/secrets.toml`.
4. Si falla, los *logs* están en el panel. Casi siempre es una dependencia que falta en
   `requirements.txt`.

**El repositorio es público.** Antes de desplegar ahí, revisa el historial de git, no solo el
último commit.

### Docker

In [ ]:
%%writefile Dockerfile
FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8501
CMD ["streamlit", "run", "app/main.py", \
     "--server.port=8501", "--server.address=0.0.0.0"]

In [ ]:
%%writefile .dockerignore
__pycache__/
*.pyc
*.log
.env
.streamlit/secrets.toml
.git
.venv/

**Las dos líneas del `.env` y del `secrets.toml` son una medida de seguridad, no una
optimización.** Sin ellas, `COPY . .` mete tus claves dentro de la imagen, y una imagen se
comparte, se sube a un registro y se descarga. La clave viaja con ella.

```bash
docker build -t mi-app-ia:latest .
docker run --rm -p 8501:8501 \
  -e AZURE_LANGUAGE_KEY=... \
  -e AZURE_LANGUAGE_ENDPOINT=... \
  -e AZURE_REGION=westeurope \
  mi-app-ia:latest
```

Las claves entran como **variables de entorno en el arranque**, no dentro de la imagen. Es la
misma idea del `.env`, aplicada al contenedor.

### Azure

Con la imagen ya construida, el camino corto es Container Apps:

```bash
az acr create -n MiRegistro -g MiGrupo -s Basic
az acr login -n MiRegistro
docker tag mi-app-ia:latest miregistro.azurecr.io/mi-app-ia:latest
docker push miregistro.azurecr.io/mi-app-ia:latest

az containerapp env create -g MiGrupo -n mi-entorno -l westeurope
az containerapp create -g MiGrupo -n mi-app-ia \
  --image miregistro.azurecr.io/mi-app-ia:latest \
  --target-port 8501 --ingress external \
  --env-vars AZURE_LANGUAGE_KEY=... AZURE_REGION=westeurope
```

En producción de verdad, las claves no van ni siquiera en `--env-vars`: van en **Key Vault**, y el
contenedor las lee con una identidad administrada, sin que exista ninguna clave que copiar. Queda
fuera de esta unidad, pero conviene saber que es el siguiente paso.

## 5. La lista de antes de entregar

Repásala con el proyecto delante. Son los puntos que se miran en PR2 y cada uno se comprueba en
menos de un minuto.

**Secretos**
- [ ] `git check-ignore -v .env` responde que está ignorado.
- [ ] `git log -p | grep -iE "(key|token|secret)\s*=\s*.{8,}"` no encuentra nada.
- [ ] `.env.example` está en el repositorio, sin valores.
- [ ] `.dockerignore` excluye `.env` y `.streamlit/secrets.toml`.

**Robustez**
- [ ] Toda llamada lleva `timeout`.
- [ ] Se reintentan 429 y 5xx, y no se reintenta ningún otro 4xx.
- [ ] Ningún fallo llega al usuario como traza.
- [ ] Con las credenciales mal, la aplicación lo dice y sigue en pie.

**Coste**
- [ ] Toda llamada al API pasa por `@st.cache_data`.
- [ ] Hay un presupuesto con alerta en el portal.
- [ ] La memoria dice cuántas llamadas hace un uso típico y cuánto ahorra la caché, medido.

**Despliegue**
- [ ] La URL funciona desde un navegador donde no has iniciado sesión.
- [ ] `requirements.txt` tiene las versiones fijadas.
- [ ] El README explica cómo levantarlo desde cero.

**Transparencia y accesibilidad**
- [ ] Hay un aviso visible de que la aplicación usa IA y de dónde se procesan los datos.
- [ ] Ningún resultado se transmite solo por color.
- [ ] La confianza de cada resultado está a la vista.

In [ ]:
# Limpieza de los ficheros generados por este cuaderno
import pathlib

for nombre in ("Dockerfile", ".dockerignore"):
    pathlib.Path(nombre).unlink(missing_ok=True)
print("Ficheros de prueba eliminados")

## Fin de los cuadernos de la unidad

Con esto tienes todo lo que piden la P2.2 y el PR2:

| Cuaderno | Qué te llevas |
|---|---|
| UD2.01 | `peticion_json()` y saber leer un JSON anidado |
| UD2.02 | El `.env` montado y la costumbre de no escribir claves |
| UD2.03 | `servicios/lenguaje.py` |
| UD2.04 | `servicios/vision.py` |
| UD2.05 | `servicios/voz.py` |
| UD2.06 | La estructura de la aplicación y cómo se presenta un resultado |
| UD2.07 | `servicios/http.py` con reintentos, la caché y el despliegue |

Lo que queda es tuyo: elegir el proyecto, construirlo y defenderlo.